In [1]:

import pandas as pd 
import mysql.connector

conn_mysql = mysql.connector.connect(
    host="localhost",
    user="root",
    password="NewPassword@123",
    database='amazon'
)

cursor_mysql = conn_mysql.cursor()
print("MySQL connection established!")




MySQL connection established!


In [2]:
df = pd.read_csv(r"C:\Users\rishi\OneDrive\Desktop\Python projects\amazon_india_2015.csv")

C:\Users\rishi\AppData\Local\Temp\ipykernel_45856\837227817.py:1: DtypeWarning: Columns (0: festival_name) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\rishi\OneDrive\Desktop\Python projects\amazon_india_2015.csv")


In [3]:
df.keys()

Index(['transaction_id', 'order_date', 'customer_id', 'product_id',
       'product_name', 'category', 'subcategory', 'brand',
       'original_price_inr', 'discount_percent', 'discounted_price_inr',
       'quantity', 'subtotal_inr', 'delivery_charges', 'final_amount_inr',
       'customer_city', 'customer_state', 'customer_tier',
       'customer_spending_tier', 'customer_age_group', 'payment_method',
       'delivery_days', 'delivery_type', 'is_prime_member', 'is_festival_sale',
       'festival_name', 'customer_rating', 'return_status', 'order_month',
       'order_year', 'order_quarter', 'product_weight_kg', 'is_prime_eligible',
       'product_rating'],
      dtype='str')

In [4]:
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

In [5]:
df['order_day']=df['order_date'].dt.day
df = pd.read_csv(r"C:\Users\rishi\OneDrive\Desktop\Python projects\amazon_india_2015.csv")

C:\Users\rishi\AppData\Local\Temp\ipykernel_45856\2443142523.py:2: DtypeWarning: Columns (0: festival_name) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\rishi\OneDrive\Desktop\Python projects\amazon_india_2015.csv")


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 33000 entries, 0 to 32999
Data columns (total 34 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   transaction_id          33000 non-null  str    
 1   order_date              33000 non-null  str    
 2   customer_id             33000 non-null  str    
 3   product_id              33000 non-null  str    
 4   product_name            33000 non-null  str    
 5   category                33000 non-null  str    
 6   subcategory             33000 non-null  str    
 7   brand                   33000 non-null  str    
 8   original_price_inr      33000 non-null  float64
 9   discount_percent        33000 non-null  float64
 10  discounted_price_inr    33000 non-null  float64
 11  quantity                33000 non-null  int64  
 12  subtotal_inr            33000 non-null  float64
 13  delivery_charges        33000 non-null  int64  
 14  final_amount_inr        33000 non-null  float64
 

In [7]:
# 1. Create PRODUCTS Table
cursor_mysql.execute("""
    CREATE TABLE IF NOT EXISTS products (
        product_id VARCHAR(50) NOT NULL,
        product_name VARCHAR(255),
        category VARCHAR(50),
        subcategory VARCHAR(50),
        brand VARCHAR(50),
        product_weight_kg DECIMAL(10, 2),
        is_prime_eligible BOOLEAN,
        product_rating DECIMAL(10, 1),
        PRIMARY KEY (product_id)
    );
""")
print("products Table created")
conn_mysql.commit()

# 2. Create CUSTOMERS Table
cursor_mysql.execute("""
    CREATE TABLE IF NOT EXISTS customers (
        customer_id VARCHAR(50) NOT NULL,
        customer_city VARCHAR(100),
        customer_state VARCHAR(100),
        customer_tier VARCHAR(50),
        customer_spending_tier VARCHAR(50),
        customer_age_group VARCHAR(50),
        PRIMARY KEY (customer_id)
    );
""")
print("customers Table created")
conn_mysql.commit()

# 3. Create TIME_DIMENSION Table
cursor_mysql.execute("""
    CREATE TABLE IF NOT EXISTS time_dimension (
        order_date DATE NOT NULL,
        order_day INT,
        order_month INT,
        order_year INT,
        order_quarter INT,
        PRIMARY KEY (order_date)
    );
""")
print("time_dimension Table created")
conn_mysql.commit()

# 4. Create TRANSACTIONS Table (The Fact Table)
cursor_mysql.execute("""
    CREATE TABLE IF NOT EXISTS transactions (
        transaction_id VARCHAR(50) NOT NULL,
        order_date DATE NOT NULL,
        customer_id VARCHAR(50) NOT NULL,
        product_id VARCHAR(50) NOT NULL,
        original_price_inr DECIMAL(10, 2),
        discount_percent DECIMAL(10, 2),
        discounted_price_inr DECIMAL(10, 2),
        quantity INT,
        subtotal_inr DECIMAL(10, 2),
        delivery_charges DECIMAL(10, 2), 
        final_amount_inr DECIMAL(10, 2),
        payment_method VARCHAR(50),                     
        delivery_days INT,
        delivery_type VARCHAR(50),              
        is_prime_member BOOLEAN,                
        is_festival_sale BOOLEAN,
        festival_name VARCHAR(50),
        customer_rating DECIMAL(5, 1),                      
        return_status VARCHAR(50),              
        
        PRIMARY KEY (transaction_id),
        
        -- Foreign Key Constraints (These will now succeed!)
        FOREIGN KEY (order_date) REFERENCES time_dimension(order_date),
        FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
        FOREIGN KEY (product_id) REFERENCES products(product_id)
    );
""")
print("transactions Table created")
conn_mysql.commit()

products Table created
customers Table created
time_dimension Table created
transactions Table created


In [8]:
df.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2015_00000001,2015-01-25,CUST_2015_00003884,PROD_000021,Samsung Galaxy S6 16GB Black,Electronics,Smartphones,Samsung,123614.29,27.91,...,True,Republic Day Sale,5.0,Delivered,1,2015,1,0.19,True,4.7
1,TXN_2015_00000002,2015-01-05,CUST_2015_00011709,PROD_000055,OnePlus OnePlus 2 16GB White,Electronics,Smartphones,OnePlus,54731.86,0.00,...,False,NaN,4.5,Delivered,1,2015,1,0.20,True,4.1
2,TXN_2015_00000003,2015-01-24,CUST_2015_00004782,PROD_000039,Samsung Galaxy Note 5 64GB Black,Electronics,Smartphones,Samsung,97644.25,46.93,...,True,Republic Day Sale,4.0,Delivered,1,2015,1,0.17,True,3.3
3,TXN_2015_00000004,2015-01-28,CUST_2015_00008105,PROD_000085,Motorola Moto G (3rd Gen) 16GB Black,Electronics,Smartphones,Motorola,21947.26,0.00,...,False,NaN,3.0,Delivered,1,2015,1,0.22,True,3.5
4,TXN_2015_00000005,2015-01-31,CUST_2015_00002955,PROD_000055,OnePlus OnePlus 2 16GB White,Electronics,Smartphones,OnePlus,54731.86,0.00,...,False,NaN,4.0,Delivered,1,2015,1,0.20,True,4.1


In [9]:
import pandas as pd
import numpy as np
import mysql.connector

# ==========================================================
# 1. MYSQL CONNECTION
# ==========================================================

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="NewPassword@123",
    database="amazon"
)

cursor = conn.cursor()

print("✅ Connected to MySQL")


# ==========================================================
# 2. LOAD DATA
# ==========================================================

df = pd.read_csv(
    r"C:\Users\rishi\OneDrive\Desktop\Python projects\amazon_india_2015.csv"
)

print("Rows:", len(df))


# ==========================================================
# 3. CLEAN DATA
# ==========================================================

# Convert order date
df["order_date"] = pd.to_datetime(
    df["order_date"],
    errors="coerce",
    format="mixed"
)

# Create time columns
df["order_day"] = df["order_date"].dt.day
df["order_month"] = df["order_date"].dt.month
df["order_year"] = df["order_date"].dt.year
df["order_quarter"] = df["order_date"].dt.quarter

# Replace NaN with None for MySQL
df = df.replace({np.nan: None})


# ==========================================================
# 4. BATCH INSERT FUNCTION
# ==========================================================

def batch_insert(records, sql, table_name, batch_size=1000):

    total = len(records)

    for start in range(0, total, batch_size):

        batch = records[start:start+batch_size]

        cursor.executemany(sql, batch)

        conn.commit()

        print(
            f"{table_name}: {min(start+batch_size,total)}/{total}"
        )


# ==========================================================
# 5. TIME DIMENSION
# ==========================================================

time_records = []

for row in df.itertuples(index=False):

    if row.order_date is None or pd.isna(row.order_date):
        continue

    time_records.append(

        (

            row.order_date.strftime("%Y-%m-%d"),

            int(row.order_day),

            int(row.order_month),

            int(row.order_year),

            int(row.order_quarter)

        )

    )


time_sql = """

INSERT IGNORE INTO time_dimension
(
order_date,
order_day,
order_month,
order_year,
order_quarter
)

VALUES
(%s,%s,%s,%s,%s)

"""

batch_insert(
    time_records,
    time_sql,
    "TIME_DIMENSION"
)


# ==========================================================
# 6. PRODUCTS
# ==========================================================

product_records = [

(

row.product_id,

row.product_name,

row.category,

row.subcategory,

row.brand,

row.product_weight_kg,

row.is_prime_eligible,

row.product_rating

)

for row in df.itertuples(index=False)

]


product_sql = """

INSERT IGNORE INTO products

(
product_id,
product_name,
category,
subcategory,
brand,
product_weight_kg,
is_prime_eligible,
product_rating
)

VALUES

(%s,%s,%s,%s,%s,%s,%s,%s)

"""

batch_insert(
    product_records,
    product_sql,
    "PRODUCTS"
)


# ==========================================================
# 7. CUSTOMERS
# ==========================================================

customer_records = [

(

row.customer_id,

row.customer_city,

row.customer_state,

row.customer_tier,

row.customer_spending_tier,

row.customer_age_group

)

for row in df.itertuples(index=False)

]


customer_sql = """

INSERT IGNORE INTO customers

(

customer_id,

customer_city,

customer_state,

customer_tier,

customer_spending_tier,

customer_age_group

)

VALUES

(%s,%s,%s,%s,%s,%s)

"""

batch_insert(
    customer_records,
    customer_sql,
    "CUSTOMERS"
)


# ==========================================================
# 8. TRANSACTIONS
# ==========================================================

transaction_records = []

for row in df.itertuples(index=False):

    transaction_records.append(

        (

        row.transaction_id,

        row.order_date.strftime("%Y-%m-%d")
        if row.order_date is not None and not pd.isna(row.order_date)
        else None,

        row.customer_id,

        row.product_id,

        row.original_price_inr,

        row.discount_percent,

        row.discounted_price_inr,

        row.quantity,

        row.subtotal_inr,

        row.delivery_charges,

        row.final_amount_inr,

        row.payment_method,

        row.delivery_days,

        row.delivery_type,

  None if pd.isna(row.is_prime_member) else row.is_prime_member,

  None if pd.isna(row.is_festival_sale) else row.is_festival_sale,

        row.festival_name,

        row.customer_rating,

        row.return_status

        )

    )


transaction_sql = """

INSERT IGNORE INTO transactions
(

transaction_id,
order_date,
customer_id,
product_id,
original_price_inr,
discount_percent,
discounted_price_inr,
quantity,
subtotal_inr,
delivery_charges,
final_amount_inr,
payment_method,
delivery_days,
delivery_type,
is_prime_member,
is_festival_sale,
festival_name,
customer_rating,
return_status

)

VALUES

(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)

"""

batch_insert(
    transaction_records,
    transaction_sql,
    "TRANSACTIONS"
)


# ==========================================================
# 9. CLOSE CONNECTION
# ==========================================================

cursor.close()
conn.close()

print("\n🎉 ETL Completed Successfully")

✅ Connected to MySQL
Rows:

C:\Users\rishi\AppData\Local\Temp\ipykernel_45856\4072402859.py:25: DtypeWarning: Columns (0: festival_name) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


 33000
TIME_DIMENSION: 1000/33000
TIME_DIMENSION: 2000/33000
TIME_DIMENSION: 3000/33000
TIME_DIMENSION: 4000/33000
TIME_DIMENSION: 5000/33000
TIME_DIMENSION: 6000/33000
TIME_DIMENSION: 7000/33000
TIME_DIMENSION: 8000/33000
TIME_DIMENSION: 9000/33000
TIME_DIMENSION: 10000/33000
TIME_DIMENSION: 11000/33000
TIME_DIMENSION: 12000/33000
TIME_DIMENSION: 13000/33000
TIME_DIMENSION: 14000/33000
TIME_DIMENSION: 15000/33000
TIME_DIMENSION: 16000/33000
TIME_DIMENSION: 17000/33000
TIME_DIMENSION: 18000/33000
TIME_DIMENSION: 19000/33000
TIME_DIMENSION: 20000/33000
TIME_DIMENSION: 21000/33000
TIME_DIMENSION: 22000/33000
TIME_DIMENSION: 23000/33000
TIME_DIMENSION: 24000/33000
TIME_DIMENSION: 25000/33000
TIME_DIMENSION: 26000/33000
TIME_DIMENSION: 27000/33000
TIME_DIMENSION: 28000/33000
TIME_DIMENSION: 29000/33000
TIME_DIMENSION: 30000/33000
TIME_DIMENSION: 31000/33000
TIME_DIMENSION: 32000/33000
TIME_DIMENSION: 33000/33000
PRODUCTS: 1000/33000
PRODUCTS: 2000/33000
PRODUCTS: 3000/33000
PRODUCTS: 4000/